In [ ]:
# Import system modules
import sys
import torch
from pathlib import Path

# Add parent directory to path for module imports
sys.path.insert(0, str(Path.cwd().parent))

# Import from prompts module
from prompts.base import build_prompt
from prompts.classifer import build_ticket_classifier
from prompts.summurize import build_ticket_classifier as build_ticket_summarizer

# Import from inference module (note: folder is named 'infrence' with typo)
from src.infrence.model import load_model, get_model_device
from src.infrence.generate import generate_text
from src.infrence.decoder import apply_temperature, apply_top_k, apply_top_p, select_next_token
from src.infrence.tokinezer import tokenize

In [ ]:
from pydantic import BaseModel

# Load the local model once
tokenizer, model = load_model()

class TicketValidationResult(BaseModel):
    is_valid: bool
    reason: str

def eval_ticket(ticket_content: str) -> TicketValidationResult:
    """
    Validate a ticket using the local LLM model.
    
    Args:
        ticket_content: The ticket text to validate
        
    Returns:
        TicketValidationResult: Contains is_valid boolean and reason string
    """
    # Build the validation prompt using the classifier
    prompt = build_ticket_classifier(ticket_content)
    
    # Generate text using the local model
    output = generate_text(
        tokenizer=tokenizer,
        model=model,
        prompt=prompt,
        max_new_token=100,
        temperature=0.7,
        top_p=0.9
    )
    
    # Parse the generated output
    output_lower = output.lower()
    
    # Determine if ticket is valid based on keywords in output
    is_valid = any(keyword in output_lower for keyword in ['valid', 'clear', 'actionable', 'good', 'yes'])
    is_invalid = any(keyword in output_lower for keyword in ['invalid', 'unclear', 'ambiguous', 'bad', 'no'])
    
    # If both keywords present or invalid keywords found, mark as invalid
    if is_invalid or (is_valid and is_invalid):
        is_valid = False
    
    # Extract reason from the generated text
    reason = output.strip()[:200]  # Take first 200 chars as reason
    
    return TicketValidationResult(is_valid=is_valid, reason=reason)

ValueError: Unrecognized configuration class <class 'transformers.models.t5.configuration_t5.T5Config'> for this kind of AutoModel: AutoModelForCausalLM.
Model type should be one of GPT2Config, AfmoeConfig, ApertusConfig, ArceeConfig, AriaTextConfig, BambaConfig, BartConfig, BertConfig, BertGenerationConfig, BigBirdConfig, BigBirdPegasusConfig, BioGptConfig, BitNetConfig, BlenderbotConfig, BlenderbotSmallConfig, BloomConfig, BltConfig, CamembertConfig, CodeGenConfig, CohereConfig, Cohere2Config, Cohere2MoeConfig, CpmAntConfig, CTRLConfig, CwmConfig, Data2VecTextConfig, DbrxConfig, DeepseekV2Config, DeepseekV3Config, DeepseekV32Config, DeepseekV4Config, DiffLlamaConfig, DogeConfig, Dots1Config, ElectraConfig, Emu3Config, ErnieConfig, Ernie4_5Config, Ernie4_5_MoeConfig, Exaone4Config, ExaoneMoeConfig, FalconConfig, FalconH1Config, FalconMambaConfig, FlexOlmoConfig, FuyuConfig, GemmaConfig, Gemma2Config, Gemma3Config, Gemma3TextConfig, Gemma3nConfig, Gemma3nTextConfig, Gemma4Config, Gemma

In [ ]:
def check_ticket_validation(result_dict):
    """
    Check if ticket validation result matches expected value.
    
    Args:
        result_dict: Dictionary with keys:
            - 'success': bool indicating if operation succeeded
            - 'expected_valid': expected is_valid value
            - 'result': TicketValidationResult object from eval_ticket
    
    Returns:
        bool: True if validation matches expectation, False otherwise
    """
    if result_dict.get('success') is False:
        return False
    
    expected_valid = result_dict.get('expected_valid')
    actual_result = result_dict.get('result')
    
    if actual_result is None:
        return False
    
    # Compare expected vs actual validation result
    return expected_valid == actual_result.is_valid

In [ ]:
# Test the validation function with sample tickets
test_tickets = [
    {
        "ticket": "I need help with my account login",
        "expected_valid": True,
        "description": "Clear and actionable ticket"
    },
    {
        "ticket": "Help",
        "expected_valid": False,
        "description": "Too vague"
    },
    {
        "ticket": "The system is down, users cannot access the dashboard",
        "expected_valid": True,
        "description": "Clear problem description"
    }
]

# Run validation on test tickets
validation_results = []
for test in test_tickets:
    try:
        validation_result = eval_ticket(test["ticket"])
        result_dict = {
            'success': True,
            'expected_valid': test['expected_valid'],
            'result': validation_result,
            'ticket': test['ticket'],
            'description': test['description']
        }
        validation_results.append(result_dict)
    except Exception as e:
        result_dict = {
            'success': False,
            'error': str(e),
            'ticket': test['ticket']
        }
        validation_results.append(result_dict)

# Calculate accuracy
if validation_results:
    accuracy_count = sum([check_ticket_validation(r) for r in validation_results if r.get('success')])
    total_success = sum([1 for r in validation_results if r.get('success')])
    
    if total_success > 0:
        validation_accuracy = accuracy_count / total_success
        print(f"Validation Accuracy: {validation_accuracy:.2%} ({accuracy_count}/{total_success})")
        print("\nDetailed Results:")
        for r in validation_results:
            if r.get('success'):
                match = "✓" if check_ticket_validation(r) else "✗"
                print(f"{match} {r['description']}: Expected={r['expected_valid']}, Got={r['result'].is_valid}")
            else:
                print(f"✗ Error: {r['error']}")
    else:
        print("No successful validations to report")